# NHANES III retina × ECG × 30-year mortality — one-click pipeline

**Study question.** In a nationally representative US sample aged ≥40 years (NHANES III, 1988–1994), is retinopathy
graded from fundus photographs (a marker of systemic microvascular disease) associated with long-term cardiovascular
and all-cause mortality (follow-up through 31 Dec 2019), and does the risk compound with electrocardiographic
evidence of myocardial substrate (ECG-LVH, major/minor Minnesota-code abnormalities, prolonged QTc, first-degree AV
block, wide QRS)?

**Why NHANES III.** It is the only free, download-and-go survey with *both* graded fundus photography (one eye,
45°, age ≥40) *and* a resting 12-lead ECG (age ≥40) in the same participants, plus NCHS linked mortality.
Retinopathy and mortality in NHANES III were examined before with follow-up to 2006 (Ricardo et al., Am J Kidney Dis
2014;64:198-203); a PubMed search (Sep 2026) found no publication combining the NHANES III retinal grading with the
NHANES III ECG file.

**Data sources (all public-use, no application):**

| File | Content used | Link |
|---|---|---|
| NHANES III Series 11 No. 1A `adult.dat` | demographics, weights/strata/PSU, questionnaire (diabetes, hypertension treatment, CVD history, smoking) | https://wwwn.cdc.gov/nchs/data/nhanes3/1a/ADULT-acc.pdf |
| `exam.dat` | BMI, blood pressure, **fundus photography grading (FPP\*)** | https://wwwn.cdc.gov/nchs/data/nhanes3/1a/exam-acc.pdf |
| `lab.dat` | HbA1c, lipids, creatinine, CRP, glucose | https://wwwn.cdc.gov/nchs/data/nhanes3/1a/lab-acc.pdf |
| Series 11 No. 2A `nh3ecg.dat` | 12-lead ECG: intervals, axes, Minnesota codes, Novacode classifications | https://wwwn.cdc.gov/nchs/data/nhanes3/2a/NH3ECG-acc.pdf |
| NCHS 2019 Public-use Linked Mortality File | vital status, leading cause of death, person-months from MEC exam | https://www.cdc.gov/nchs/data-linkage/mortality-public.htm |

**How to run.** Put this notebook and `nhanes3_tools.py` in one folder and run all cells. Raw files are downloaded
to `./data/` (≈150 MB, first run only) and every table/figure is written to `./output/`.

**Two safety gates.** The pipeline never guesses a code. It (1) prints the official variable labels for every
variable it uses and (2) prints value counts for every coded variable. The core analysis (retinopathy → mortality)
runs with the documented coding of the fundus summary scores; the ECG classification flags (ECPG1–ECPG6) use the
coding documented in `NH3ECG-acc.pdf` (0 = absent, 1 = present, 8 = blank but applicable → missing).

In [ ]:
# ------------------------------------------------------------------ 0. Setup
import sys, json, warnings, platform
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HERE = Path.cwd()
sys.path.insert(0, str(HERE))
if not (HERE / "nhanes3_tools.py").exists():
    raise FileNotFoundError("nhanes3_tools.py must sit in the same folder as this notebook")
import nhanes3_tools as nt
import lifelines

DATA_DIR = HERE / "data"
OUT_DIR = HERE / "output"
FIG_DIR = OUT_DIR / "figures"
for d in (DATA_DIR, OUT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings("ignore", category=FutureWarning)
from lifelines.exceptions import StatisticalWarning
warnings.filterwarnings("ignore", category=StatisticalWarning)          # sampling weights in KM are intended
from statsmodels.tools.sm_exceptions import SpecificationWarning
warnings.filterwarnings("ignore", category=SpecificationWarning)       # freq_weights + cluster SE (approximation noted)
pd.set_option("display.width", 160, "display.max_columns", 50, "display.max_rows", 200)
nt.set_figure_style()

RUNLOG = {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
          "lifelines": lifelines.__version__}
print(RUNLOG)

## CONFIG — variable map and coding decisions

Names marked `# confirmed` were read from the official SAS programs/documentation. Names marked `# verify` exist in
the files but their exact wording/coding is printed in the label-check cell below — read that output once and adjust
here if needed. Nothing else in the notebook hard-codes a variable name.

In [ ]:
# ------------------------------------------------------------------ 1. CONFIG
VARS = {
    "adult": {
        "HSAGEIR": "age at interview (years)",                     # confirmed
        "HSSEX": "sex (1 = male, 2 = female)",                      # confirmed label 'Sex'; coding printed below
        "DMARETHN": "race-ethnicity",                               # confirmed
        "HFA8R": "highest grade or year of school completed",       # confirmed
        "WTPFEX6": "total MEC-examined sample final weight",        # confirmed
        "SDPPSU6": "pseudo-PSU",                                    # confirmed
        "SDPSTRA6": "pseudo-stratum",                               # confirmed
        "HAB1": "self-rated general health",                        # confirmed
        "HAD1": "doctor ever told you have diabetes",               # verify
        "HAD6": "now taking insulin",                               # verify
        "HAD10": "now taking diabetes pills",                       # verify
        "HAE2": "ever told high blood pressure",                    # verify
        "HAE5A": "now taking prescribed medicine for high BP",      # verify
        "HAF10": "ever told you had a heart attack",                # verify
        "HAR1": "smoked at least 100 cigarettes in life",           # verify
        "HAR3": "smoke cigarettes now",                             # verify
    },
    "exam": {
        "BMPBMI": "body mass index (kg/m2)",                        # verify
        "PEPMNK1R": "average systolic BP (K1), mmHg",               # verify
        "PEPMNK5R": "average diastolic BP (K5), mmHg",              # verify
        "FPPPHOTO": "fundus photo taken",                           # verify
        "FPPEYE": "eye photographed (1 = right, 2 = left)",         # verify
        "FPP1060": "gradability of fundus photograph",              # confirmed (06 = entire field ungradable)
        "FPP1070": "diabetic retinopathy level (ETDRS scale)",      # confirmed
        "FPP1230": "hard drusen",                                   # confirmed
        "FPP1240": "soft drusen",                                   # confirmed
        "FPPSURET": "summary diabetic retinopathy score",           # confirmed
        "FPPSUMAC": "summary age-related maculopathy score",        # confirmed
        "FPPSUDRU": "summary drusen score",                         # confirmed
    },
    "lab": {
        "GHP": "glycated hemoglobin (%)",                           # confirmed
        "G1P": "plasma glucose (mg/dL)",                            # confirmed
        "TCP": "serum total cholesterol (mg/dL)",                   # confirmed
        "HDP": "serum HDL cholesterol (mg/dL)",                     # confirmed
        "TGP": "serum triglycerides (mg/dL)",                       # confirmed
        "CEP": "serum creatinine (mg/dL)",                          # confirmed
        "CRP": "serum C-reactive protein (mg/dL)",                  # confirmed
    },
    "ecg": {
        "ECPRATE": "heart rate (bpm)",                              # confirmed
        "ECPPR": "PR interval (ms)",                                # confirmed
        "ECPQRS": "QRS duration (ms)",                              # confirmed
        "ECPQT": "QT interval (ms)",                                # confirmed
        "ECPAXIS1": "P axis", "ECPAXIS2": "QRS axis", "ECPAXIS3": "T axis",   # confirmed
        "ECPBEAT": "rhythm code",                                   # confirmed
        "ECPG1": "major ECG abnormalities",                         # confirmed
        "ECPG2": "minor ECG abnormalities",                         # confirmed
        "ECPG3": "probable myocardial infarction",                  # confirmed
        "ECPG4": "possible myocardial infarction",                  # confirmed
        "ECPG5": "probable left ventricular hypertrophy",           # confirmed
        "ECPG6": "possible LVH by Minnesota code",                  # confirmed
        "ECPLVM": "ECG-estimated LV mass", "ECPLVMI": "ECG-estimated LV mass index",  # confirmed
    },
}

# Coding decisions --------------------------------------------------------------------------
# Fundus summary scores: documented in the NHANES III multiple-imputation release (imp*-acc.pdf):
#   FPPSURET 0 = none, 1 = mild NPDR, 2 = moderate/severe NPDR, 3 = proliferative; any other value -> missing
#   FPPSUMAC 0 = none, 1 = early ARM, 2 = late ARM
#   FPPSUDRU 0 = none, 1 = questionable, 2 = present
FPP_RET_CODES = {0: 0, 1: 1, 2: 2, 3: 3}
FPP_ARM_CODES = {0: 0, 1: 1, 2: 2}
FPP_DRU_CODES = {0: 0, 1: 1, 2: 2}
# FPPSURET derivation (imp*-acc.pdf): none = FPP1070 010-015; mild = 020 microaneurysms only / 031 early NPDR;
# moderate/severe = 041/051; proliferative = 060-070, or photocoagulation outside arcades (FPP1214 = 02) with
# FPP1070 other than 012. FPP1070 graded levels in the audit: 10, 11, 12, 14, 15, 20, 31, 41, 51, 60, 65, 70
# (88 = can't grade, 888 = blank but applicable).

# Retinopathy exposure definition:
#   "any_ge12"     NHANES III literature convention (Lin et al., PLoS One 2016): FPP1070 level 12 (non-diabetic
#                  retinopathy), 14-15 (hemorrhages, soft/hard exudates or IRMA without microaneurysms), 20
#                  (microaneurysms only), 31/41/51 (non-proliferative), 60/65/70 (proliferative), plus laser-treated
#                  eyes (FPPSURET = 3). Level 12 is grouped with "mild" in severity analyses.
#   "any_ge14"     same without level 12 = NHANES 2005-2008 OPDUARMA rule ("levels >= 14"); severity grouped like
#                  OPDURL4 (none 10-13, mild 14-31, moderate/severe 41-51, proliferative 60+).
#   "summary_ge20" NCHS summary score FPPSURET >= 1 (levels 10-15 counted as none).
RET_PRIMARY = "any_ge12"

# Published NHANES III benchmarks (age >= 40, gradable photographs, survey-weighted):
#   Keenan et al., Am J Ophthalmol 2009: 5.4% of all persons, 3.5% of persons without diabetes.
#   Lin et al., PLoS One 2016 (subsample with liver ultrasound, n = 5,963; "any_ge12" definition): 5.3% overall,
#   3.6% / 3.5% in the two groups without diabetes (without / with NAFLD).
BENCHMARK_RET_PREV = {"Keenan 2009 all": 5.4, "Keenan 2009 no diabetes": 3.5,
                      "Lin 2016 all": 5.3, "Lin 2016 no diabetes (no NAFLD / NAFLD)": "3.6 / 3.5"}

# Questionnaire yes/no coding (NHANES III convention 1 = yes, 2 = no; 8/9 = blank/don't know) - checked below
YES, NO = 1, 2

# ECG classification codes (ECPG1..ECPG6), from NH3ECG-acc.pdf: 0 = absent, 1 = present,
# 8 = blank but applicable (-> missing). Documented frequencies, e.g. ECPG1 6,991 / 1,550 / 20.
# Set both to None to skip the ECG-joint analyses; the retinopathy -> mortality core analysis still runs.
ECG_YES_CODE = 1
ECG_NO_CODE = 0

# Optional NHANES III creatinine calibration to IDMS-traceable values (NHANES III creatinine was not
# standardised). Leave None until you have checked the equation in the NHANES III / CKD-EPI analytic notes.
CREATININE_CAL = None      # e.g. {"intercept": -0.184, "slope": 0.960}

# Outcome definitions (NCHS UCOD_LEADING): 1 = diseases of heart; 5 = cerebrovascular diseases
CVD_CODES = {1, 5}
HEART_CODES = {1}

AGE_MIN = 40
MIN_FOLLOWUP_YEARS_SENS = 2.0   # sensitivity analysis: exclude deaths within the first 2 years

## 1. Download the public-use files (first run only)

In [ ]:
paths = nt.fetch_all(DATA_DIR)
for k, v in paths.items():
    for kind, p in v.items():
        print(f"{k:6s} {kind:4s} {p.name:35s} {p.stat().st_size/1e6:8.1f} MB")
print("\nDocumentation PDFs (open these to verify codes):")
for k, v in nt.NHANES3_FILES.items():
    print(f"  {k:6s} {v['doc']}")

## 2. Parse the official layouts and labels; check every mapped variable

In [ ]:
layouts, labels = {}, {}
for k in ("adult", "exam", "lab", "ecg"):
    sas_text = paths[k]["sas"].read_text(encoding="latin-1", errors="replace")
    layouts[k] = nt.parse_sas_input(sas_text)
    labels[k] = nt.parse_sas_labels(sas_text)
    print(f"{k:6s}: {len(layouts[k]):5d} fields in INPUT, {len(labels[k]):5d} labels; "
          f"first={layouts[k][0].name}, last={layouts[k][-1].name}, LRECL>={max(f.end for f in layouts[k])}")
    pd.DataFrame([(f.name, f.start, f.end, f.char, f.dec, labels[k].get(f.name, "")) for f in layouts[k]],
                 columns=["variable", "start", "end", "char", "dec", "label"]
                 ).to_csv(OUT_DIR / f"layout_{k}.csv", index=False)

# --- label check: every mapped variable must exist; print its official label -----------------------
problems = []
rows = []
for k, vmap in VARS.items():
    names = {f.name for f in layouts[k]}
    for v, meaning in vmap.items():
        ok = v in names
        rows.append((k, v, meaning, labels[k].get(v, "(no LABEL statement parsed)"), ok))
        if not ok:
            problems.append(f"{k}:{v}")
label_check = pd.DataFrame(rows, columns=["file", "variable", "assumed meaning", "official label", "found"])
label_check.to_csv(OUT_DIR / "variable_label_check.csv", index=False)
print(label_check.to_string(index=False))
if problems:
    print("\n!! Variables missing from layouts:", problems)
    print("   Use nt.find_vars(labels['adult'], 'keyword') to locate the right name, then fix CONFIG.")

In [ ]:
# Helper searches to confirm the 'verify' items (edit keywords freely)
for k, kws in [("adult", ["diabetes"]), ("adult", ["heart attack"]), ("adult", ["stroke"]),
               ("adult", ["heart failure"]), ("adult", ["blood pressure"]), ("adult", ["smok"]),
               ("adult", ["cigarette"]), ("exam", ["systolic"]), ("exam", ["diastolic"]),
               ("exam", ["body mass"]), ("exam", ["retinopathy"]), ("exam", ["maculopathy"]),
               ("exam", ["drusen"]), ("exam", ["gradab"])]:
    hits = nt.find_vars(labels[k], *kws)
    print(f"--- {k}: {kws}  ({len(hits)} hits)")
    print(hits.head(25).to_string(index=False) if len(hits) else "   (none)")

## 3. Load the variables, merge on SEQN, define the analytic sample

In [ ]:
adult = nt.read_fixed(paths["adult"]["dat"], layouts["adult"], keep=["SEQN"] + list(VARS["adult"]))
exam  = nt.read_fixed(paths["exam"]["dat"],  layouts["exam"],  keep=["SEQN"] + list(VARS["exam"]))
lab   = nt.read_fixed(paths["lab"]["dat"],   layouts["lab"],   keep=["SEQN"] + list(VARS["lab"]))
ecg   = nt.read_fixed(paths["ecg"]["dat"],   layouts["ecg"],   keep=["SEQN"] + list(VARS["ecg"]))
mort  = nt.read_lmf(paths["mort"]["dat"])
for name, d in [("adult", adult), ("exam", exam), ("lab", lab), ("ecg", ecg), ("mort", mort)]:
    assert d["SEQN"].is_unique, f"SEQN not unique in {name}"
    print(f"{name:6s} rows={len(d):6d}")

df = (adult.merge(exam, on="SEQN", how="left")
           .merge(lab, on="SEQN", how="left")
           .merge(ecg, on="SEQN", how="left", indicator="ecg_merge")
           .merge(mort, on="SEQN", how="left", indicator="mort_merge"))
df["has_ecg"] = (df["ecg_merge"] == "both").astype(int)
df["has_mort"] = (df["mort_merge"] == "both").astype(int)
df = df.drop(columns=["ecg_merge", "mort_merge"])

# --- flow of the analytic sample --------------------------------------------------------------------
flow = []
def step(label, mask):
    flow.append((label, int(mask.sum())))
    print(f"{label:60s} n = {int(mask.sum()):6d}")
    return mask

m = step("Adult interview file", pd.Series(True, index=df.index))
m = step(f"Age >= {AGE_MIN} at interview", m & (df["HSAGEIR"] >= AGE_MIN))
m = step("MEC-examined (WTPFEX6 > 0)", m & (df["WTPFEX6"] > 0))
m = step("Fundus summary retinopathy score present", m & df["FPPSURET"].isin(FPP_RET_CODES.keys()))
m = step("Mortality-linkage eligible (ELIGSTAT = 1)", m & (df["ELIGSTAT"] == 1))
m = step("Follow-up time present (PERMTH_EXM)", m & df["PERMTH_EXM"].notna())
cohort_mask = m.copy()
m_ecg = step("... and ECG file record present", m & (df["has_ecg"] == 1))
pd.DataFrame(flow, columns=["step", "n"]).to_csv(OUT_DIR / "sample_flow.csv", index=False)

## 4. Value-count audit of every coded variable (read this once)

In [ ]:
audit_vars = ["HSSEX", "DMARETHN", "HAD1", "HAD6", "HAD10", "HAE2", "HAE5A", "HAF10", "HAR1", "HAR3",
              "FPPPHOTO", "FPPEYE", "FPP1060", "FPP1070", "FPP1230", "FPP1240",
              "FPPSURET", "FPPSUMAC", "FPPSUDRU",
              "ECPBEAT", "ECPG1", "ECPG2", "ECPG3", "ECPG4", "ECPG5", "ECPG6",
              "ELIGSTAT", "MORTSTAT", "UCOD_LEADING"]
sub = df[df["HSAGEIR"] >= AGE_MIN]
audit_rows = []
for v in audit_vars:
    if v not in sub:
        continue
    vc = sub[v].value_counts(dropna=False).sort_index()
    for val, n in vc.items():
        audit_rows.append((v, val, int(n)))
    print(f"\n{v} — {next((labels[k][v] for k in labels if v in labels[k]), '')}")
    print(vc.to_string())
pd.DataFrame(audit_rows, columns=["variable", "value", "n_age40plus"]).to_csv(OUT_DIR / "value_count_audit.csv", index=False)
print("\nContinuous ECG/lab summaries (age >= 40):")
print(sub[[c for c in ["ECPRATE", "ECPPR", "ECPQRS", "ECPQT", "ECPLVM", "ECPLVMI", "GHP", "CEP", "CRP", "BMPBMI",
                       "PEPMNK1R", "PEPMNK5R"] if c in sub]].describe().T.round(2))

## 5. Derived variables

Primary exposure (CONFIG `RET_PRIMARY = "any_ge12"`): the NHANES III literature convention (Lin et al., PLoS One
2016) — `FPP1070` level 12 (non-diabetic retinopathy), 14–15 (hemorrhages, exudates or IRMA without microaneurysms),
20 (microaneurysms only), 31–51 (non-proliferative), 60–70 (proliferative), plus laser-treated eyes. Sensitivity
definitions: level ≥ 14 (the NHANES 2005–2008 `OPDUARMA` rule, level 12 counted as none) and the NCHS summary score
`FPPSURET` ≥ 1 (levels 10–15 counted as none). The first table compares the survey-weighted prevalence of each
definition with the published NHANES III estimates; the second tabulates lesion patterns among exposed participants.

In [ ]:
d = df[cohort_mask].copy()

# --- exposures ---------------------------------------------------------------------------------------
lvl = d["FPP1070"].where(d["FPP1070"].between(10, 70))        # ETDRS-based level; 88 / 888 -> missing
summ = d["FPPSURET"].map(FPP_RET_CODES)                         # NCHS summary score 0-3
# Harmonised 4-level severity (NHANES 2005-2008 OPDURL4 grouping); proliferative includes laser-treated eyes (summ == 3)
d["ret4_ge14"] = np.select([summ == 3, summ == 2, (summ == 1) | lvl.isin([14, 15]), summ == 0],
                           [3.0, 2.0, 1.0, 0.0], default=np.nan)
d["ret_any_ge14"] = np.where(np.isnan(d["ret4_ge14"]), np.nan, (d["ret4_ge14"] >= 1).astype(float))
d["ret4_ge20"] = summ.astype(float)
d["ret_any_ge20"] = np.where(summ.isna(), np.nan, (summ >= 1).astype(float))
# NHANES III literature convention: also count level 12 ("non-diabetic retinopathy"), grouped with mild for severity
d["ret_any_ge12"] = np.where(np.isnan(d["ret4_ge14"]), np.nan, ((d["ret4_ge14"] >= 1) | (lvl == 12)).astype(float))
d["ret4_ge12"] = np.where(np.isnan(d["ret4_ge14"]), np.nan,
                          np.where((d["ret4_ge14"] == 0) & (lvl == 12), 1.0, d["ret4_ge14"]))
PRIMARY_COLS = {"any_ge12": ("ret4_ge12", "ret_any_ge12"), "any_ge14": ("ret4_ge14", "ret_any_ge14"),
                "summary_ge20": ("ret4_ge20", "ret_any_ge20")}
if RET_PRIMARY not in PRIMARY_COLS:
    raise ValueError(f"RET_PRIMARY must be one of {list(PRIMARY_COLS)}")
sc, an = PRIMARY_COLS[RET_PRIMARY]
assert d[sc].notna().all(), "retinopathy exposure missing inside the analytic cohort"
d["ret_score"] = d[sc].astype(int)
d["ret_any"] = d[an].astype(int)
d["ret_sev"] = pd.cut(d["ret_score"], bins=[-1, 0, 1, 3], labels=["none", "mild", "moderate_or_worse"])
print(f"Primary retinopathy definition: {RET_PRIMARY}")
print("Cross-tab FPPSURET (rows) x FPP1070 level (columns), analytic cohort:")
print(pd.crosstab(summ, d["FPP1070"], dropna=False))
d["arm_score"] = d["FPPSUMAC"].map(FPP_ARM_CODES)
d["arm_any"] = np.where(d["arm_score"].isna(), np.nan, (d["arm_score"] >= 1).astype(float))
d["drusen_score"] = d["FPPSUDRU"].map(FPP_DRU_CODES)

# --- covariates --------------------------------------------------------------------------------------
d["age"] = d["HSAGEIR"]
d["female"] = (d["HSSEX"] == 2).astype(int)
d["race_eth"] = d["DMARETHN"].astype("Int64")
d["edu_years"] = d["HFA8R"].where(d["HFA8R"] <= 17)          # 88/99 codes -> missing (verify in ADULT-acc.pdf)
d["bmi"] = d["BMPBMI"].where(d["BMPBMI"].between(10, 80))
d["sbp"] = d["PEPMNK1R"].where(d["PEPMNK1R"].between(60, 300))
d["dbp"] = d["PEPMNK5R"].where(d["PEPMNK5R"].between(20, 200))
d["hba1c"] = d["GHP"].where(d["GHP"].between(2, 20))
d["tc"] = d["TCP"].where(d["TCP"].between(50, 800))
d["hdl"] = d["HDP"].where(d["HDP"].between(5, 200))
d["crp"] = d["CRP"].where(d["CRP"] >= 0)
scr = d["CEP"].where(d["CEP"].between(0.1, 20))
if CREATININE_CAL:
    scr = CREATININE_CAL["intercept"] + CREATININE_CAL["slope"] * scr
d["egfr"] = nt.egfr_ckdepi2021(scr, d["age"], d["female"] == 1)

d["dm_told"] = (d["HAD1"] == YES).astype(int)
d["diabetes"] = ((d["dm_told"] == 1) | (d["hba1c"] >= 6.5) | (d["HAD6"] == YES) | (d["HAD10"] == YES)).astype(int)
d["bp_meds"] = (d["HAE5A"] == YES).astype(int)
d["hypertension"] = ((d["sbp"] >= 140) | (d["dbp"] >= 90) | (d["bp_meds"] == 1)).astype(int)
d["smoke"] = pd.Series(np.select([d["HAR1"] == NO, (d["HAR1"] == YES) & (d["HAR3"] == YES), (d["HAR1"] == YES) & (d["HAR3"] == NO)],
                                 ["never", "current", "former"], default="missing"), index=d.index).replace({"missing": np.nan})
d["prior_mi"] = (d["HAF10"] == YES).astype(int)          # extend with stroke / CHF items once mapped

# --- ECG ---------------------------------------------------------------------------------------------
d["hr"] = d["ECPRATE"].where(d["ECPRATE"].between(30, 200))
d["pr_ms"] = d["ECPPR"].where(d["ECPPR"].between(60, 400))
d["qrs_ms"] = d["ECPQRS"].where(d["ECPQRS"].between(40, 250))
d["qt_ms"] = d["ECPQT"].where(d["ECPQT"].between(200, 700))
d = pd.concat([d, nt.qtc_all(d["qt_ms"], d["hr"])], axis=1)
d["qtc_prolonged"] = np.where(d["QTc_Bazett"].isna(), np.nan,
                              (((d["female"] == 1) & (d["QTc_Bazett"] >= 460)) | ((d["female"] == 0) & (d["QTc_Bazett"] >= 450))).astype(float))
d["av_block1"] = np.where(d["pr_ms"].isna(), np.nan, (d["pr_ms"] >= 200).astype(float))
d["wide_qrs"] = np.where(d["qrs_ms"].isna(), np.nan, (d["qrs_ms"] >= 120).astype(float))
if ECG_YES_CODE is not None and ECG_NO_CODE is not None:
    for v, new in [("ECPG1", "ecg_major"), ("ECPG2", "ecg_minor"), ("ECPG3", "ecg_probable_mi"),
                   ("ECPG4", "ecg_possible_mi"), ("ECPG5", "ecg_probable_lvh"), ("ECPG6", "ecg_possible_lvh")]:
        # present -> 1, absent -> 0, any other code (8 = blank but applicable) or no ECG record -> missing
        d[new] = np.select([d[v] == ECG_YES_CODE, d[v] == ECG_NO_CODE], [1.0, 0.0], default=np.nan)
    d["ecg_any_major_or_lvh"] = np.where(d[["ecg_major", "ecg_probable_lvh"]].isna().all(axis=1), np.nan,
                                         ((d["ecg_major"] == 1) | (d["ecg_probable_lvh"] == 1)).astype(float))
else:
    print("ECG_YES_CODE/ECG_NO_CODE not set -> Novacode/Minnesota classification flags not derived.")

# --- outcomes ----------------------------------------------------------------------------------------
d["time_y"] = d["PERMTH_EXM"] / 12.0
d["death_all"] = (d["MORTSTAT"] == 1).astype(int)
d["death_cvd"] = ((d["MORTSTAT"] == 1) & d["UCOD_LEADING"].isin(CVD_CODES)).astype(int)
d["death_heart"] = ((d["MORTSTAT"] == 1) & d["UCOD_LEADING"].isin(HEART_CODES)).astype(int)
d.loc[d["time_y"] <= 0, "time_y"] = 0.5 / 12.0     # deaths recorded in the exam month: half-month convention

d["w"] = d["WTPFEX6"]
d["psu"] = d["SDPSTRA6"].fillna(-1).astype(int).astype(str) + "_" + d["SDPPSU6"].fillna(-1).astype(int).astype(str)
DESIGN_PSUS = d[["SDPSTRA6", "psu"]].drop_duplicates()
SVY = dict(cluster="psu", strata_col="SDPSTRA6", design_psus=DESIGN_PSUS)   # design-based variance: strata + PSU
print(f"Survey design: {d['SDPSTRA6'].nunique()} strata, {d['psu'].nunique()} PSUs")
from lifelines import KaplanMeierFitter
fu_rkm = KaplanMeierFitter().fit(d["time_y"], 1 - d["death_all"]).median_survival_time_   # reverse Kaplan-Meier
print(f"Analytic cohort n = {len(d)}; deaths = {d['death_all'].sum()}; CVD deaths = {d['death_cvd'].sum()}; "
      f"heart-disease deaths = {d['death_heart'].sum()}")
print(f"Median follow-up (reverse Kaplan-Meier) = {fu_rkm:.1f} y; median observed time (incl. deaths) = {d['time_y'].median():.1f} y")
print("Retinopathy, primary definition (unweighted):", d["ret_any"].value_counts().to_dict(),
      "| weighted prevalence = {:.1f}%".format(100 * nt.wprop(d["ret_any"], d["w"])))

# --- prevalence of each definition versus the published NHANES III estimate --------------------------
defs = [("Levels 12, 14-15, >= 20 or laser (NHANES III convention)", "ret_any_ge12"),
        ("Levels >= 14 or laser (NHANES 2005-2008 OPDUARMA rule)", "ret_any_ge14"),
        ("FPPSURET >= 1 (levels >= 20 or laser)", "ret_any_ge20")]
rows = []
for lab, col in defs:
    rows.append({"definition": lab, "column": col, "primary": col == PRIMARY_COLS[RET_PRIMARY][1],
                 "n_exposed": int(d[col].sum()),
                 "weighted_pct_all": round(100 * nt.wprop(d[col], d["w"]), 2),
                 "weighted_pct_no_diabetes": round(100 * nt.wprop(d.loc[d["diabetes"] == 0, col], d.loc[d["diabetes"] == 0, "w"]), 2),
                 "weighted_pct_diabetes": round(100 * nt.wprop(d.loc[d["diabetes"] == 1, col], d.loc[d["diabetes"] == 1, "w"]), 2)})
ret_defs = pd.DataFrame(rows)
ret_defs.to_csv(OUT_DIR / "retinopathy_definitions.csv", index=False)
print(ret_defs.to_string(index=False))
print("Published NHANES III benchmarks (weighted %):", BENCHMARK_RET_PREV,
      "- samples and diabetes definitions differ slightly from ours")

# --- lesion patterns among participants with retinopathy (NHANES III convention), by diabetes status ----------
pattern = pd.Series(np.select(
    [summ == 3, lvl.isin([31, 41, 51]), lvl == 20, lvl.isin([14, 15]), lvl == 12],
    ["Proliferative or laser-treated", "Non-proliferative (levels 31-51)", "Microaneurysms only (level 20)",
     "Hemorrhage, exudates or IRMA without microaneurysms (levels 14-15)", "Non-diabetic retinopathy (level 12)"],
    default="none"), index=d.index)
prow = []
for dm, lab in [(0, "no diabetes"), (1, "diabetes"), (None, "all")]:
    sub = d[(d["ret_any_ge12"] == 1) & ((d["diabetes"] == dm) if dm is not None else True)]
    for pat in pattern[sub.index].unique():
        prow.append({"group": lab, "pattern": pat, "n": int((pattern[sub.index] == pat).sum()),
                     "weighted_pct": round(100 * nt.wprop((pattern[sub.index] == pat).astype(float), sub["w"]), 1)})
lesions = pd.DataFrame(prow).sort_values(["group", "weighted_pct"], ascending=[True, False])
lesions.to_csv(OUT_DIR / "retinopathy_lesion_patterns.csv", index=False)
print(lesions.to_string(index=False))
d.to_csv(OUT_DIR / "analytic_dataset.csv", index=False)

## 6. Table 1 (survey-weighted) by retinopathy status

In [ ]:
d["smoke_current"] = np.where(d["smoke"].isna(), np.nan, (d["smoke"] == "current").astype(float))
cont = ["age", "bmi", "sbp", "dbp", "hba1c", "tc", "hdl", "egfr", "hr", "pr_ms", "qrs_ms", "QTc_Bazett"]
binv = ["female", "diabetes", "hypertension", "bp_meds", "smoke_current", "prior_mi", "av_block1", "wide_qrs", "qtc_prolonged"]
if "ecg_major" in d:
    binv += ["ecg_major", "ecg_minor", "ecg_probable_lvh"]
t1 = nt.table1(d, by="ret_any", w="w", continuous=cont, binary=binv, categorical=["race_eth"])
t1.to_csv(OUT_DIR / "table1_by_retinopathy.csv", index=False)
t1

## 7. Retinopathy → long-term mortality (weighted Cox, PSU-clustered robust SE)

Model 1: age, sex, race-ethnicity. Model 2: + education, smoking, BMI, SBP, BP medication, diabetes, HbA1c,
total/HDL cholesterol, eGFR, prior MI. Model 3: Model 2 + ECG abnormalities (tests whether the retinal signal is
independent of the myocardial substrate).

In [ ]:
def prep(dd):
    # Explicit dummy variables; missing categories stay missing (complete-case in each model).
    x = dd.copy()
    # DMARETHN: 1 = non-Hispanic white (reference), 2 = non-Hispanic black, 3 = Mexican-American, 4 = other  (verify in ADULT-acc.pdf)
    for lev in sorted(x["race_eth"].dropna().unique()):
        if lev == x["race_eth"].dropna().min():
            continue
        x[f"race_eth_{int(lev)}"] = np.where(x["race_eth"].isna(), np.nan, (x["race_eth"] == lev).astype(float))
    # smoking: never = reference
    for lev in ["former", "current"]:
        x[f"smoke_{lev}"] = np.where(x["smoke"].isna(), np.nan, (x["smoke"] == lev).astype(float))
    return x

race_cols = lambda x: [c for c in x.columns if c.startswith("race_eth_")]
smoke_cols = lambda x: ["smoke_former", "smoke_current"]
X = prep(d)
M1 = ["age", "female"] + race_cols(X)
M2 = M1 + ["edu_years", "bmi", "sbp", "bp_meds", "diabetes", "hba1c", "tc", "hdl", "egfr", "prior_mi"] + smoke_cols(X)
M3 = M2 + [c for c in ["ecg_major", "ecg_probable_lvh", "av_block1", "wide_qrs", "qtc_prolonged"] if c in X]

results = []
for outcome in ["death_all", "death_cvd", "death_heart"]:
    for mname, covs in [("M1", M1), ("M2", M2), ("M3", M3)]:
        for expo in ["ret_any"]:
            try:
                cph, res = nt.cox_weighted(X, "time_y", outcome, [expo] + covs, "w", **SVY)
                r = res.loc[expo]
                results.append({"outcome": outcome, "model": mname, "exposure": expo, "n": res.attrs["n"],
                                "events": res.attrs["events"], "HR": r["HR"], "lo95": r["HR_lo95"], "hi95": r["HR_hi95"], "p": r["p"]})
            except Exception as e:
                results.append({"outcome": outcome, "model": mname, "exposure": expo, "error": str(e)[:120]})
cox_main = pd.DataFrame(results)
cox_main.to_csv(OUT_DIR / "cox_retinopathy_mortality.csv", index=False)
cox_main.round(3)

In [ ]:
# Severity (ordinal trend) and stratification by diabetes status ---------------------------------------
X["ret_score_c"] = X["ret_score"].astype(float)
X["ret_mild"] = (X["ret_score"] == 1).astype(float)
X["ret_modplus"] = (X["ret_score"] >= 2).astype(float)
sev = []
for outcome in ["death_all", "death_cvd"]:
    _, res = nt.cox_weighted(X, "time_y", outcome, ["ret_score_c"] + M2, "w", **SVY)
    sev.append({"outcome": outcome, "term": "per severity level (trend)", **res.loc["ret_score_c"].to_dict(), "n": res.attrs["n"], "events": res.attrs["events"]})
    _, res = nt.cox_weighted(X, "time_y", outcome, ["ret_mild", "ret_modplus"] + M2, "w", **SVY)
    for t in ["ret_mild", "ret_modplus"]:
        sev.append({"outcome": outcome, "term": t, **res.loc[t].to_dict(), "n": res.attrs["n"], "events": res.attrs["events"]})
    for dm, lab in [(0, "no diabetes"), (1, "diabetes")]:
        sub = X[X["diabetes"] == dm]
        covs = [c for c in M2 if c not in ("diabetes",)]
        _, res = nt.cox_weighted(sub, "time_y", outcome, ["ret_any"] + covs, "w", **SVY)
        sev.append({"outcome": outcome, "term": f"ret_any | {lab}", **res.loc["ret_any"].to_dict(), "n": res.attrs["n"], "events": res.attrs["events"]})
cox_sev = pd.DataFrame(sev)
cox_sev.to_csv(OUT_DIR / "cox_severity_and_diabetes_strata.csv", index=False)
cox_sev.round(3)

## 8. Joint retinal–ECG phenotype (runs when `ECG_YES_CODE` / `ECG_NO_CODE` are set)

Four mutually exclusive groups: neither / retinopathy only / ECG abnormality only / both. Reported: adjusted HRs,
multiplicative interaction, and the relative excess risk due to interaction (RERI, additive scale).

In [ ]:
joint_ok = "ecg_any_major_or_lvh" in X.columns
if joint_ok:
    XJ = X[X["ecg_any_major_or_lvh"].notna()].copy()
    XJ["grp"] = np.select([(XJ["ret_any"] == 0) & (XJ["ecg_any_major_or_lvh"] == 0),
                           (XJ["ret_any"] == 1) & (XJ["ecg_any_major_or_lvh"] == 0),
                           (XJ["ret_any"] == 0) & (XJ["ecg_any_major_or_lvh"] == 1),
                           (XJ["ret_any"] == 1) & (XJ["ecg_any_major_or_lvh"] == 1)],
                          ["neither", "retinopathy_only", "ecg_only", "both"], default=None)
    for g in ["retinopathy_only", "ecg_only", "both"]:
        XJ[f"grp_{g}"] = (XJ["grp"] == g).astype(float)
    XJ["ret_x_ecg"] = XJ["ret_any"] * XJ["ecg_any_major_or_lvh"]
    joint = []
    for outcome in ["death_all", "death_cvd"]:
        _, res = nt.cox_weighted(XJ, "time_y", outcome, ["grp_retinopathy_only", "grp_ecg_only", "grp_both"] + M2, "w", **SVY)
        for t in ["grp_retinopathy_only", "grp_ecg_only", "grp_both"]:
            joint.append({"outcome": outcome, "term": t, **res.loc[t].to_dict(), "n": res.attrs["n"], "events": res.attrs["events"]})
        terms = ["grp_retinopathy_only", "grp_ecg_only", "grp_both"]
        hr10, hr01, hr11 = (res.loc[t, "HR"] for t in terms)
        reri = hr11 - hr10 - hr01 + 1
        grad = np.array([-hr10, -hr01, hr11])                       # d RERI / d (b10, b01, b11)
        se_reri = float(np.sqrt(grad @ res.attrs["vcov"].loc[terms, terms].values @ grad))
        joint.append({"outcome": outcome, "term": "RERI (HR11 - HR10 - HR01 + 1)", "HR": reri,
                      "HR_lo95": reri - 1.959964 * se_reri, "HR_hi95": reri + 1.959964 * se_reri})
        _, res_i = nt.cox_weighted(XJ, "time_y", outcome, ["ret_any", "ecg_any_major_or_lvh", "ret_x_ecg"] + M2, "w", **SVY)
        joint.append({"outcome": outcome, "term": "multiplicative interaction (ret x ecg)", **res_i.loc["ret_x_ecg"].to_dict()})
    cox_joint = pd.DataFrame(joint)
    cox_joint.to_csv(OUT_DIR / "cox_joint_retina_ecg.csv", index=False)
    print(XJ["grp"].value_counts())
    display(cox_joint.round(3))
else:
    print("Skipped: set ECG_YES_CODE / ECG_NO_CODE in CONFIG after checking NH3ECG-acc.pdf and the value-count audit.")

## 9. Cross-sectional: retinopathy and electrocardiographic intervals

Survey-weighted linear models (PSU-clustered robust SE) for QTc, PR and QRS as functions of retinopathy, adjusted for Model 2
covariates and heart rate where appropriate; logistic models for the dichotomous ECG flags.

In [ ]:
import statsmodels.api as sm
cs = []
Xc = X.copy()
for y in ["QTc_Fridericia", "QTc_Bazett", "pr_ms", "qrs_ms"]:
    cols = ["ret_any"] + M2 + (["hr"] if y in ("pr_ms", "qrs_ms") else [])
    dd = Xc[[y, "w", "psu"] + cols].dropna()
    if len(dd) < 50:
        continue
    model = sm.WLS(dd[y], sm.add_constant(dd[cols]), weights=dd["w"] / dd["w"].mean())
    fit = model.fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(dd["psu"])[0]})
    cs.append({"outcome": y, "type": "weighted linear", "beta_ret_any": fit.params["ret_any"],
               "lo95": fit.conf_int().loc["ret_any", 0], "hi95": fit.conf_int().loc["ret_any", 1], "p": fit.pvalues["ret_any"], "n": int(fit.nobs)})
for y in [c for c in ["qtc_prolonged", "av_block1", "wide_qrs", "ecg_major", "ecg_probable_lvh"] if c in Xc]:
    cols = ["ret_any"] + M2
    dd = Xc[[y, "w", "psu"] + cols].dropna()
    if dd[y].nunique() < 2 or len(dd) < 50 or pd.crosstab(dd["ret_any"], dd[y]).min().min() < 5:
        print(f"{y}: skipped (sparse cells in retinopathy x outcome table)")
        continue
    glm = sm.GLM(dd[y], sm.add_constant(dd[cols]), family=sm.families.Binomial(), freq_weights=dd["w"] / dd["w"].mean())
    fit = glm.fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(dd["psu"])[0]})
    cs.append({"outcome": y, "type": "weighted logistic (OR)", "beta_ret_any": np.exp(fit.params["ret_any"]),
               "lo95": np.exp(fit.conf_int().loc["ret_any", 0]), "hi95": np.exp(fit.conf_int().loc["ret_any", 1]), "p": fit.pvalues["ret_any"], "n": int(fit.nobs)})
cross = pd.DataFrame(cs)
cross.to_csv(OUT_DIR / "crosssectional_retinopathy_ecg.csv", index=False)
cross.round(3)

## 10. Sensitivity analyses

In [ ]:
sens = []
# (a) exclude prior MI at baseline
sub = X[X["prior_mi"] == 0]
for outcome in ["death_all", "death_cvd"]:
    _, res = nt.cox_weighted(sub, "time_y", outcome, ["ret_any"] + [c for c in M2 if c != "prior_mi"], "w", **SVY)
    sens.append({"analysis": "exclude prior MI", "outcome": outcome, **res.loc["ret_any"].to_dict(), "n": res.attrs["n"], "events": res.attrs["events"]})
# (b) exclude the first MIN_FOLLOWUP_YEARS_SENS years (reverse causation)
sub = X[X["time_y"] > MIN_FOLLOWUP_YEARS_SENS]
for outcome in ["death_all", "death_cvd"]:
    _, res = nt.cox_weighted(sub, "time_y", outcome, ["ret_any"] + M2, "w", **SVY)
    sens.append({"analysis": f"follow-up > {MIN_FOLLOWUP_YEARS_SENS} y", "outcome": outcome, **res.loc["ret_any"].to_dict(), "n": res.attrs["n"], "events": res.attrs["events"]})
# (c) ARM as an alternative retinal exposure
sub = X[X["arm_any"].notna()]
for outcome in ["death_all", "death_cvd"]:
    _, res = nt.cox_weighted(sub, "time_y", outcome, ["arm_any"] + M2, "w", **SVY)
    sens.append({"analysis": "exposure = any ARM", "outcome": outcome, **res.loc["arm_any"].to_dict(), "n": res.attrs["n"], "events": res.attrs["events"]})
# (d) alternative retinopathy definitions (the two that are not primary)
for lab, col in [("exposure = levels 12, 14-15, >= 20 or laser (NHANES III convention)", "ret_any_ge12"),
                 ("exposure = levels >= 14 or laser (OPDUARMA rule)", "ret_any_ge14"),
                 ("exposure = FPPSURET >= 1 (levels >= 20 or laser)", "ret_any_ge20")]:
    if col == PRIMARY_COLS[RET_PRIMARY][1]:
        continue
    for outcome in ["death_all", "death_cvd"]:
        _, res = nt.cox_weighted(X, "time_y", outcome, [col] + M2, "w", **SVY)
        sens.append({"analysis": lab, "outcome": outcome, **res.loc[col].to_dict(), "n": res.attrs["n"], "events": res.attrs["events"]})
# (e) unweighted Cox
for outcome in ["death_all", "death_cvd"]:
    X1 = X.assign(one=1.0)
    _, res = nt.cox_weighted(X1, "time_y", outcome, ["ret_any"] + M2, "one", **SVY, normalize_weights=False)
    sens.append({"analysis": "unweighted", "outcome": outcome, **res.loc["ret_any"].to_dict(), "n": res.attrs["n"], "events": res.attrs["events"]})
sens = pd.DataFrame(sens)
sens.to_csv(OUT_DIR / "sensitivity_analyses.csv", index=False)
sens.round(3)

## 10b. Robustness checks for interpretation

(a) Missingness of Model 2 covariates. (b) Nested models refitted on common complete-case samples, so that the change
from Model 1 to Models 2–3 reflects adjustment rather than sample loss, plus sequential adjustment on the Model 2
sample. (c) Effect modification by diabetes (product term). (d) Proportional hazards: period-specific hazard ratios
(0–10, 10–20, > 20 years; persons alive at the start of each period) with a design-based heterogeneity test, and an
unweighted Schoenfeld-residual test.
Results for the primary definition and for `FPPSURET ≥ 1` are written to `output/robustness_checks.csv`.

In [ ]:
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test

rob = []
def _add(check, outcome, model, term, res_row, n=None, events=None):
    rob.append({"check": check, "outcome": outcome, "model": model, "term": term,
                "HR": res_row["HR"], "HR_lo95": res_row["HR_lo95"], "HR_hi95": res_row["HR_hi95"], "p": res_row["p"],
                "n": n, "events": events})

# (a) missingness of Model 2 covariates -----------------------------------------------------------------
miss = (X[M2].isna().mean() * 100).round(2).sort_values(ascending=False)
miss.rename("pct_missing").to_csv(OUT_DIR / "missingness_model2.csv")
print("Percent missing among Model 2 covariates (analytic cohort):")
print(miss[miss > 0].to_string() if (miss > 0).any() else "none")

# (b) common complete-case samples and sequential adjustment -------------------------------------------
cc2 = X.dropna(subset=M2)
cc3 = X.dropna(subset=M3)
BLOCKS = {"M1 + diabetes, HbA1c": M1 + ["diabetes", "hba1c"],
          "M1 + SBP, BP medication": M1 + ["sbp", "bp_meds"],
          "M1 + other M2 covariates": M1 + [c for c in M2 if c not in M1 + ["diabetes", "hba1c", "sbp", "bp_meds"]]}
for outcome in ["death_all", "death_cvd"]:
    for samp_name, samp, models in [("common sample M2", cc2, [("M1", M1), ("M2", M2)] + list(BLOCKS.items())),
                                    ("common sample M3", cc3, [("M1", M1), ("M2", M2), ("M3", M3)])]:
        for mname, covs in models:
            _, res = nt.cox_weighted(samp, "time_y", outcome, ["ret_any"] + covs, "w", **SVY)
            _add(samp_name, outcome, mname, "ret_any", res.loc["ret_any"], res.attrs["n"], res.attrs["events"])

# (c) effect modification by diabetes -------------------------------------------------------------------
for col in ["ret_any", "ret_any_ge20"]:
    Z = X.copy()
    Z["ret_x_dm"] = Z[col] * Z["diabetes"]
    for outcome in ["death_all", "death_cvd"]:
        _, res = nt.cox_weighted(Z, "time_y", outcome, [col, "ret_x_dm"] + M2, "w", **SVY)
        _add(f"interaction x diabetes ({col})", outcome, "M2", "ret_x_dm", res.loc["ret_x_dm"], res.attrs["n"], res.attrs["events"])

# (d1) period-specific hazard ratios: persons alive at the start of each period, time re-origined at the period
#      start (same risk sets as left truncation) and censored at the period end; heterogeneity = joint Wald test
#      using stacked influence values (accounts for the overlap of persons across periods)
PERIODS = [(0, 10), (10, 20), (20, 40)]
def period_frame(data, outcome, a, b):
    P = data[data["time_y"] > a].copy()
    P["t_p"] = P["time_y"].clip(upper=b) - a
    P["ev_p"] = ((P[outcome] == 1) & (P["time_y"] <= b)).astype(int)
    return P
for col in ["ret_any", "ret_any_ge20"]:
    for outcome in ["death_all", "death_cvd"]:
        betas, infl = [], []
        for a, b in PERIODS:
            _, res = nt.cox_weighted(period_frame(X, outcome, a, b), "t_p", "ev_p", [col] + M2, "w", **SVY)
            _add(f"period {a}-{b if b < 40 else 'end'} y ({col})", outcome, "M2", col, res.loc[col], res.attrs["n"], res.attrs["events"])
            betas.append(res.loc[col, "beta"]); infl.append(res.attrs["influence"][col])
        het = nt.design_wald(betas, infl, X["SDPSTRA6"], X["psu"], np.array([[1, -1, 0], [0, 1, -1]]), design_psus=DESIGN_PSUS)
        rob.append({"check": f"heterogeneity across the three periods ({col})", "outcome": outcome, "model": "M2", "term": col,
                    "HR": np.nan, "HR_lo95": np.nan, "HR_hi95": np.nan, "p": het["p"], "n": None, "events": None})

# (d2) Schoenfeld-residual test (unweighted diagnostic) -----------------------------------------------------
for outcome in ["death_all", "death_cvd"]:
    dd = X[["time_y", outcome, "ret_any"] + M2].dropna()
    cph = CoxPHFitter().fit(dd, duration_col="time_y", event_col=outcome)
    t = proportional_hazard_test(cph, dd, time_transform="rank")
    rob.append({"check": "Schoenfeld test, rank time (unweighted)", "outcome": outcome, "model": "M2", "term": "ret_any",
                "HR": np.nan, "HR_lo95": np.nan, "HR_hi95": np.nan, "p": float(t.summary.loc["ret_any", "p"]),
                "n": len(dd), "events": int(dd[outcome].sum())})

robust_df = pd.DataFrame(rob)
robust_df.to_csv(OUT_DIR / "robustness_checks.csv", index=False)
robust_df.round(3)

## 10c. Time-varying association

The proportional-hazards assumption fails for retinopathy, so the single 30-year hazard ratio averages a changing
association. This section reports (a) hazard ratios in 5-year windows with a design-based heterogeneity test,
(b) the first decade after examination (0–10 years; deaths after 10 years censored): nested models including ECG
findings, alternative exposure definitions, severity, diabetes strata and interaction, the joint retina–ECG phenotype
and, as exploratory, lesion types; and (c) crude survey-weighted 10-year absolute risks (all-cause: Kaplan–Meier;
CVD: Aalen–Johansen with non-CVD death as a competing event). Model 2 unless stated.

In [ ]:
from lifelines import KaplanMeierFitter, AalenJohansenFitter
warnings.filterwarnings("ignore", message="Tied event times were detected")   # AJ jitters ties (seeded)

tv, errors = [], []
def _tv(analysis, outcome, term, res, extra=None):
    r = res.loc[term]
    row = {"analysis": analysis, "outcome": outcome, "term": term, "HR": r["HR"], "HR_lo95": r["HR_lo95"],
           "HR_hi95": r["HR_hi95"], "p": r["p"], "n": res.attrs["n"], "events": res.attrs["events"]}
    row.update(extra or {})
    tv.append(row)

def _fit(analysis, outcome, data, terms, covs):
    try:
        _, res = nt.cox_weighted(data, "t_p", "ev_p", terms + covs, "w", **SVY)
        for t in terms:
            _tv(analysis, outcome, t, res)
        return res
    except Exception as e:                       # sparse cells are recorded, never silently dropped
        errors.append({"analysis": analysis, "outcome": outcome, "error": str(e)[:200]})
        return None

# (a) 5-year windows ----------------------------------------------------------------------------------------
WINDOWS = [(0, 5), (5, 10), (10, 15), (15, 20), (20, 25), (25, 40)]
Cdiff = np.eye(len(WINDOWS))[:-1] - np.eye(len(WINDOWS), k=1)[:-1]          # successive differences
for outcome in ["death_all", "death_cvd"]:
    betas, infl = [], []
    for a, b in WINDOWS:
        _, res = nt.cox_weighted(period_frame(X, outcome, a, b), "t_p", "ev_p", ["ret_any"] + M2, "w", **SVY)
        _tv(f"window {a}-{b if b < 40 else 'end'} y", outcome, "ret_any", res, {"window_start": a, "window_end": min(b, 31)})
        betas.append(res.loc["ret_any", "beta"]); infl.append(res.attrs["influence"]["ret_any"])
    het = nt.design_wald(betas, infl, X["SDPSTRA6"], X["psu"], Cdiff, design_psus=DESIGN_PSUS)
    tv.append({"analysis": "heterogeneity across 5-year windows", "outcome": outcome, "term": "ret_any", "p": het["p"]})

# (b) first decade after examination ------------------------------------------------------------------------
LES = {"les_level12": "Non-diabetic retinopathy (level 12)",
       "les_14_15": "Hemorrhage, exudates or IRMA without microaneurysms (levels 14-15)",
       "les_ma": "Microaneurysms only (level 20)",
       "les_npdr": "Non-proliferative (levels 31-51)",
       "les_pdr": "Proliferative or laser-treated"}
for k, v in LES.items():
    X[k] = (pattern.reindex(X.index) == v).astype(float)
alt_defs = [c for c in ["ret_any_ge12", "ret_any_ge14", "ret_any_ge20"] if c != PRIMARY_COLS[RET_PRIMARY][1]]
for outcome in ["death_all", "death_cvd"]:
    FD = lambda data: period_frame(data, outcome, 0, 10)
    for mname, covs in [("M1", M1), ("M2", M2), ("M3", M3)]:
        _fit(f"first decade, common M3 sample, {mname}", outcome, FD(cc3), ["ret_any"], covs)
    _fit("first decade, M2", outcome, FD(X), ["ret_any"], M2)
    for col in alt_defs:
        _fit(f"first decade, M2, {col}", outcome, FD(X), [col], M2)
    _fit("first decade, M2, severity", outcome, FD(X), ["ret_mild", "ret_modplus"], M2)
    for dm, lab in [(0, "no diabetes"), (1, "diabetes")]:
        _fit(f"first decade, M2, {lab}", outcome, FD(X[X["diabetes"] == dm]), ["ret_any"], [c for c in M2 if c != "diabetes"])
    _fit("first decade, M2, interaction x diabetes", outcome, FD(X.assign(ret_x_dm=X["ret_any"] * X["diabetes"])),
         ["ret_x_dm"], ["ret_any"] + M2)
    if joint_ok:
        jt = ["grp_retinopathy_only", "grp_ecg_only", "grp_both"]
        rj = _fit("first decade, M2, joint retina-ECG", outcome, FD(XJ), jt, M2)
        if rj is not None and rj.loc[jt, "HR"].notna().all():
            hr10, hr01, hr11 = (rj.loc[t, "HR"] for t in jt)
            reri = hr11 - hr10 - hr01 + 1
            grad = np.array([-hr10, -hr01, hr11])
            se_reri = float(np.sqrt(grad @ rj.attrs["vcov"].loc[jt, jt].values @ grad))
            tv.append({"analysis": "first decade, M2, joint retina-ECG", "outcome": outcome,
                       "term": "RERI (HR11 - HR10 - HR01 + 1)", "HR": reri,
                       "HR_lo95": reri - 1.959964 * se_reri, "HR_hi95": reri + 1.959964 * se_reri})
    _fit("first decade, M2, lesion type (exploratory)", outcome, FD(X), list(LES), M2)
    # reverse-causation check: exclude deaths in the first 2 years (persons alive at 2 y, followed to 10 y)
    _fit("2-10 y after examination, M2", outcome, period_frame(X, outcome, 2, 10), ["ret_any"], M2)

tv_df = pd.DataFrame(tv)
tv_df.to_csv(OUT_DIR / "time_varying_analyses.csv", index=False)
if errors:
    pd.DataFrame(errors).to_csv(OUT_DIR / "time_varying_fit_errors.csv", index=False)
    print("Models that could not be fitted (see time_varying_fit_errors.csv):", len(errors))

# (c) crude survey-weighted 10-year absolute risks ------------------------------------------------------------
ar = []
for g in (0, 1):
    sub = d[d["ret_any"] == g]
    w = sub["w"] / sub["w"].mean()
    km = KaplanMeierFitter().fit(sub["time_y"].values, sub["death_all"].values, weights=w.values)
    risk_all = 1 - float(km.survival_function_at_times(10).iloc[0])
    ev = np.select([sub["death_cvd"] == 1, sub["death_all"] == 1], [1, 2], default=0)
    aj = AalenJohansenFitter(seed=2026, calculate_variance=False).fit(sub["time_y"].values, ev, event_of_interest=1, weights=w.values)
    cif = aj.cumulative_density_
    ar.append({"retinopathy": g, "n": len(sub), "risk_all_cause_10y_pct": round(100 * risk_all, 2),
               "risk_cvd_10y_pct": round(100 * float(cif[cif.index <= 10].iloc[-1, 0]), 2)})
ar = pd.DataFrame(ar)
ar.to_csv(OUT_DIR / "absolute_risk_10y.csv", index=False)
print(ar.to_string(index=False))

# (d) E-values for the first-decade Model 2 estimates (VanderWeele & Ding 2017) ----------------------------------
w_all = d["w"] / d["w"].mean()
risk10_all = 1 - float(KaplanMeierFitter().fit(d["time_y"].values, d["death_all"].values, weights=w_all.values)
                       .survival_function_at_times(10).iloc[0])
ev_all = np.select([d["death_cvd"] == 1, d["death_all"] == 1], [1, 2], default=0)
cif_all = AalenJohansenFitter(seed=2026, calculate_variance=False).fit(
    d["time_y"].values, ev_all, event_of_interest=1, weights=w_all.values).cumulative_density_
risk10_cvd = float(cif_all[cif_all.index <= 10].iloc[-1, 0])
evs = []
for outcome, risk10 in [("death_all", risk10_all), ("death_cvd", risk10_cvd)]:
    common = risk10 > 0.15                      # outcome common by 10 years -> HR converted to approximate RR
    sel = tv_df[(tv_df["analysis"] == "first decade, M2") & (tv_df["outcome"] == outcome)]
    for _, r in sel.iterrows():
        e_est, e_ci = nt.evalue_hr(r["HR"], r["HR_lo95"], r["HR_hi95"], common_outcome=common)
        evs.append({"outcome": outcome, "HR": r["HR"], "HR_lo95": r["HR_lo95"], "HR_hi95": r["HR_hi95"],
                    "weighted_10y_risk_pct": round(100 * risk10, 1), "common_outcome_conversion": common,
                    "E_value_estimate": round(e_est, 2), "E_value_CI_limit": round(e_ci, 2)})
evalues = pd.DataFrame(evs)
evalues.to_csv(OUT_DIR / "evalues_first_decade.csv", index=False)
print(evalues.to_string(index=False))

tv_df = pd.DataFrame(tv)                          # includes RERI rows added above
tv_df.to_csv(OUT_DIR / "time_varying_analyses.csv", index=False)
tv_df.round(3)

## 11. Figures (submission style: Arial, white background, no gridlines, black/grey; panel letters A/B)

Figure 1: survey-weighted cumulative mortality by retinopathy (A all-cause, Kaplan–Meier; B CVD, Aalen–Johansen).
Figure 2: hazard ratios for retinopathy in 5-year windows (Model 2; A all-cause, B CVD).
Figure S1: overall 30-year hazard ratios, Models 1–3.

In [ ]:
from lifelines import KaplanMeierFitter, AalenJohansenFitter
warnings.filterwarnings("ignore", message="Tied event times were detected")   # AJ jitters ties (seeded)

# Figure 1 ---------------------------------------------------------------------------------------------------
import matplotlib.ticker as mticker
AT_RISK_TIMES = [0, 5, 10, 15, 20, 25, 30]
fig, axes = plt.subplots(1, 2, figsize=(11, 5.6))
for ax, outcome, letter in [(axes[0], "death_all", "A"), (axes[1], "death_cvd", "B")]:
    ymax_panel = 0.0
    for g, col, lab in [(0, nt.GREYS[2], "No retinopathy"), (1, nt.GREYS[0], "Retinopathy")]:
        sub = d[d["ret_any"] == g]
        w = sub["w"] / sub["w"].mean()
        if outcome == "death_all":
            km = KaplanMeierFitter().fit(sub["time_y"].values, sub["death_all"].values, weights=w.values)
            curve = 1 - km.survival_function_.iloc[:, 0]
        else:
            ev = np.select([sub["death_cvd"] == 1, sub["death_all"] == 1], [1, 2], default=0)
            curve = AalenJohansenFitter(seed=2026, calculate_variance=False).fit(
                sub["time_y"].values, ev, event_of_interest=1, weights=w.values).cumulative_density_.iloc[:, 0]
        ax.step(curve.index, curve.values, where="post", color=col, linewidth=2, label=lab)
        ymax_panel = max(ymax_panel, float(curve[curve.index <= 31].max()))
    ax.axvline(10, color=nt.GREYS[1], linestyle="--", linewidth=1, label="10 years after examination")
    ax.set_xlabel("Years since examination")
    ax.set_ylabel("Cumulative mortality" if outcome == "death_all" else "Cumulative CVD mortality")
    ax.set_xlim(0, 31); ax.set_ylim(0, ymax_panel * 1.25)
    ax.set_xticks(AT_RISK_TIMES)
    ax.legend(loc="upper left", frameon=False)
fig.subplots_adjust(left=0.15, right=0.98, top=0.92, bottom=0.27, wspace=0.30)
# number at risk (unweighted), aligned with the x-axis ticks of each panel
for ax in axes:
    ax.text(0.0, -0.20, "No. at risk", transform=ax.transAxes, ha="left", va="top", fontsize=10, fontweight="bold")
    for row, (g, lab) in enumerate([(0, "No retinopathy"), (1, "Retinopathy")]):
        sub = d[d["ret_any"] == g]
        yrow = -0.27 - 0.07 * row
        for t in AT_RISK_TIMES:
            ax.text(t, yrow, f"{int((sub['time_y'] >= t).sum()):,}", transform=ax.get_xaxis_transform(),
                    ha="center", va="top", fontsize=9, color=nt.GREYS[0] if g == 1 else nt.GREYS[1])
        if ax is axes[0]:
            ax.text(-0.07, yrow, lab, transform=ax.transAxes, ha="right", va="top", fontsize=9,
                    color=nt.GREYS[0] if g == 1 else nt.GREYS[1])
nt.align_panel_letters(fig, [(axes[0], "A"), (axes[1], "B")])
fig.savefig(FIG_DIR / "fig1_cumulative_mortality_by_retinopathy.png", dpi=300)
fig.savefig(FIG_DIR / "fig1_cumulative_mortality_by_retinopathy.pdf")
plt.close(fig)

# Figure 2 ---------------------------------------------------------------------------------------------------
win = tv_df[tv_df["analysis"].str.startswith("window")].copy()
lo_all, hi_all = win["HR_lo95"].min(), win["HR_hi95"].max()
ticks = [2.0 ** k for k in range(-8, 6) if lo_all / 1.3 <= 2.0 ** k <= hi_all * 1.3] or [1.0]
fig, axes = plt.subplots(1, 2, figsize=(11, 5.0), sharey=True)
for ax, outcome in [(axes[0], "death_all"), (axes[1], "death_cvd")]:
    w_ = win[win["outcome"] == outcome].reset_index(drop=True)
    xpos = np.arange(len(w_))
    ax.errorbar(xpos, w_["HR"], yerr=[w_["HR"] - w_["HR_lo95"], w_["HR_hi95"] - w_["HR"]], fmt="s", color="black",
                ecolor="black", capsize=3, markersize=6, label="Hazard ratio (95% CI)")
    ax.axhline(1.0, color=nt.GREYS[2], linestyle="--", linewidth=1, label="HR = 1")
    for xi, hr in zip(xpos, w_["HR"]):
        ax.annotate(f"{hr:.2f}", (xi, hr), xytext=(8, 0), textcoords="offset points", ha="left", va="center", fontsize=10)
    ax.set_yscale("log")
    ax.set_ylim(lo_all / 1.3, hi_all * 1.3)
    ax.yaxis.set_major_locator(mticker.FixedLocator(ticks))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))
    ax.yaxis.set_minor_locator(mticker.NullLocator())
    ax.set_xticks(xpos)
    ax.set_xticklabels([f"{int(a)}–{int(b)}" for a, b in zip(w_["window_start"], w_["window_end"])])
    ax.set_xlim(-0.5, len(w_) - 0.3)
    ax.set_xlabel("Years since examination")
axes[0].set_ylabel("Hazard ratio")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 0.0))
fig.tight_layout(rect=(0, 0.08, 1, 0.96))
nt.align_panel_letters(fig, [(axes[0], "A"), (axes[1], "B")])
fig.savefig(FIG_DIR / "fig2_hazard_ratio_by_5year_window.png", dpi=300)
fig.savefig(FIG_DIR / "fig2_hazard_ratio_by_5year_window.pdf")
plt.close(fig)

# Figure S1: overall models ----------------------------------------------------------------------------------
fp = cox_main.dropna(subset=["HR"]).copy() if "HR" in cox_main else pd.DataFrame()
if len(fp):
    fp["label"] = fp["outcome"].map({"death_all": "All-cause", "death_cvd": "CVD", "death_heart": "Heart disease"}) + " — " + fp["model"]
    fig, ax = plt.subplots(figsize=(7.0, 0.45 * len(fp) + 1.6))
    y = np.arange(len(fp))[::-1]
    ax.errorbar(fp["HR"], y, xerr=[fp["HR"] - fp["lo95"], fp["hi95"] - fp["HR"]], fmt="s", color="black", ecolor="black", capsize=3, markersize=5,
                label="Hazard ratio (95% CI)")
    ax.axvline(1.0, color=nt.GREYS[2], linewidth=1, linestyle="--", label="HR = 1")
    ax.set_xscale("log")
    lo_x, hi_x = fp["lo95"].min() / 1.1, fp["hi95"].max() * 1.1
    ax.set_xlim(lo_x, hi_x)
    xt = [v for v in [0.5, 0.67, 0.8, 1.0, 1.25, 1.5, 2.0, 3.0, 4.0] if lo_x <= v <= hi_x]
    ax.xaxis.set_major_locator(mticker.FixedLocator(xt))
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))
    ax.xaxis.set_minor_locator(mticker.NullLocator())
    for yi, hr, lo, hi in zip(y, fp["HR"], fp["lo95"], fp["hi95"]):
        ax.annotate(f"{hr:.2f} ({lo:.2f}–{hi:.2f})", (1.02, yi), xycoords=("axes fraction", "data"),
                    va="center", ha="left", fontsize=10)
    ax.set_yticks(y); ax.set_yticklabels(fp["label"])
    ax.set_xlabel("Hazard ratio")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "figS1_forest_overall_models.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIG_DIR / "figS1_forest_overall_models.pdf", bbox_inches="tight")
    plt.close(fig)
print(sorted(p.name for p in FIG_DIR.iterdir()))

## 12. Run log

Variance: all Cox models use design-based Taylor-linearisation variance (pseudo-strata + pseudo-PSUs;
`nhanes3_tools.cox_weighted`), the estimator used by R `survey::svycoxph`. Subgroup and period analyses keep every
PSU of the design (domain estimation). On synthetic stratified test data the implementation reproduced lifelines'
cluster-robust SEs exactly and agreed with a delete-one-PSU jackknife to within 0.1%.

In [ ]:
RUNLOG.update({
    "n_cohort": int(len(d)), "deaths_all": int(d["death_all"].sum()), "deaths_cvd": int(d["death_cvd"].sum()),
    "median_followup_reverse_km_years": float(fu_rkm), "ecg_joint_analysis_run": bool(joint_ok),
    "retinopathy_primary_definition": RET_PRIMARY, "variance": "design-based Taylor linearisation (strata + PSU)",
    "config": {"AGE_MIN": AGE_MIN, "ECG_YES_CODE": ECG_YES_CODE, "ECG_NO_CODE": ECG_NO_CODE, "CREATININE_CAL": CREATININE_CAL,
               "CVD_CODES": sorted(CVD_CODES)},
})
(OUT_DIR / "run_log.json").write_text(json.dumps(RUNLOG, indent=2))
print(json.dumps(RUNLOG, indent=2))
print("\nOutputs:", sorted(p.name for p in OUT_DIR.iterdir()))